## Example 1: Lossy compression at a fixed error tolerance

In [ ]:
import numpy as np
import zfpy

rng = np.random.default_rng(42)
field = np.cumsum(rng.standard_normal((128, 128)), axis=1)  # a smooth 2-D field
packed = zfpy.compress_numpy(field, tolerance=1e-4)
restored = zfpy.decompress_numpy(packed)
error = float(np.abs(field - restored).max())
print(f"zfpy from {zfpy.__file__}")
print(f"{field.nbytes} bytes -> {len(packed)} bytes, max error {error:.2e}")
assert restored.shape == field.shape and restored.dtype == field.dtype
assert error <= 1e-4, error
assert len(packed) < field.nbytes

## Example 2: Lossless (reversible) mode round-trips exactly

In [ ]:
import numpy as np
import zfpy

rng = np.random.default_rng(1)
for dtype in (np.float32, np.float64, np.int32, np.int64):
    data = (rng.standard_normal((16, 16, 16)) * 1000).astype(dtype)
    restored = zfpy.decompress_numpy(zfpy.compress_numpy(data))
    print(dtype.__name__, data.shape, "exact:", np.array_equal(restored, data))
    assert restored.dtype == data.dtype and np.array_equal(restored, data)

## Example 3: Fixed-rate and fixed-precision modes

In [ ]:
import numpy as np
import zfpy

data = np.linspace(0.0, 1.0, 64 * 64, dtype=np.float64).reshape(64, 64) ** 2
by_rate = zfpy.compress_numpy(data, rate=8)  # about 8 bits per value
by_precision = zfpy.compress_numpy(data, precision=20)
print("rate=8:", len(by_rate), "bytes; precision=20:", len(by_precision), "bytes")
assert abs(len(by_rate) - data.size * 8 / 8) <= 64  # fixed rate: the size follows the rate
for packed in (by_rate, by_precision):
    restored = zfpy.decompress_numpy(packed)
    assert restored.shape == data.shape
    assert float(np.abs(restored - data).max()) < 1e-2

## Example 4: One to four dimensions

In [ ]:
import numpy as np
import zfpy

rng = np.random.default_rng(7)
for shape in [(4096,), (64, 64), (16, 16, 16), (8, 8, 8, 8)]:
    data = rng.random(shape)
    restored = zfpy.decompress_numpy(zfpy.compress_numpy(data, tolerance=1e-6))
    print(f"{len(shape)}-D {shape}: max error {float(np.abs(data - restored).max()):.1e}")
    assert restored.shape == data.shape
    assert float(np.abs(data - restored).max()) <= 1e-6

## Example 5: The ZFPY codec in numcodecs (Zarr's compressor)

In [ ]:
import numpy as np
import numcodecs
from numcodecs import get_codec
from numcodecs.zfpy import ZFPY

codec = ZFPY(mode=4, tolerance=1e-3)  # mode 4 = fixed accuracy
data = np.random.default_rng(3).standard_normal((32, 32)).astype(np.float32)
encoded = codec.encode(data)
decoded = codec.decode(encoded).reshape(data.shape)
print("numcodecs", numcodecs.__version__, "->", len(encoded), "bytes for", data.nbytes)
assert float(np.abs(decoded - data).max()) <= 1e-3
again = get_codec(codec.get_config())  # what Zarr does when it reads an array's metadata
assert type(again) is ZFPY and again.get_config() == codec.get_config()